# Comparative analysis

In [ ]:
import os
import numpy as np
import matplotlib.pyplot as plt
import pandas as pd

from utils import load_analysis

### Paths and config (modify only these lines)

In [ ]:
pd.set_option('display.max_rows', None)
analysis_path = os.path.join(
    '..', 'results', 'analyses', 'comparison_analysis.obj'
)

### Load (and display table of) comparison analysis

In [ ]:
# Load and display all table
analysis_raw = load_analysis(analysis_path)
analysis = pd.DataFrame(analysis_raw)
analysis['training_efficiency'] = analysis['elapsed_time'] / analysis['epochs']
analysis

In [ ]:
def get_value_and_ci_str(value, ci):
    exponent = np.floor(np.log10(value))
    base_value = value / 10**(exponent)
    base_ci = ci / 10**(exponent)
    output_str = '$(%1.2f \pm %1.2f) \\times 10^{%d}$' % (
        base_value, base_ci, exponent
    )
    return output_str

In [ ]:
for exp_name in ('rod', 'gaussian', 'shield'):
    # select current experiment with EYS initialization
    cond_exp = (analysis['experiment'] == exp_name)
    cond_init_eys = (analysis['initialization'] == 'eys')
    sae_analysis = analysis[cond_exp * cond_init_eys]
    act_name_tuple_base = tuple(sae_analysis['act_name'])
    act_sharpness_tuple_base = tuple(sae_analysis['act_sharpness'])
    act_sharpness_list = list()
    # Set act name and sharpness indices
    for (sharpness, act_name) in zip(
        act_sharpness_tuple_base, act_name_tuple_base
    ):
        param_name = '\\alpha' if act_name == 'leaky' else '\\theta'
        act_sharpness_list.append(
            '$\\angle(' + param_name + ') = ' + str(sharpness) + '$'
        )
    act_name_tuple = [
        '$\\text{LeakyReLU}_{\\alpha,5/4}$' 
        if elem == 'leaky' else '$\\text{HypAct}_{\\theta}$'
        for elem in act_name_tuple_base
    ]
    model_names_tuple = tuple(
        ['$\\textsf{' + elem.upper() + '}$' 
         for elem in sae_analysis['model_name']]
    )
    sae_analysis.pop('act_name')
    sae_analysis = sae_analysis[sae_analysis.columns.dropna()]
    sae_analysis.index = pd.MultiIndex.from_tuples(
        tuple(zip(act_name_tuple, act_sharpness_list, model_names_tuple)))
    # Get values and 5% CI as string
    sae_analysis.columns.map(''.join)
    sae_analysis['\\textbf{MSE}'] = [
        get_value_and_ci_str(value, ci)
        for value, ci in zip(sae_analysis['mse'], sae_analysis['mse_5%_ci'])
    ]
    sae_analysis['\\textbf{MRE}'] = [
        get_value_and_ci_str(value, ci)
        for value, ci in zip(sae_analysis['mre'], sae_analysis['mre_5%_ci'])
    ]
    # Pop values
    sae_analysis.pop('mse')
    sae_analysis.pop('mse_5%_ci')
    sae_analysis.pop('mre')
    sae_analysis.pop('mre_5%_ci')
    sae_analysis.pop('act_sharpness')
    sae_analysis.pop('model_name')
    sae_analysis.pop('act_parameter')
    sae_analysis.pop('elapsed_time')
    sae_analysis.pop('monitors')
    sae_analysis.pop('epochs')
    sae_analysis.pop('training_efficiency')
    sae_analysis.pop('initialization')
    sae_analysis.pop('experiment')
    # Print latex table
    print('\n\n\n% Tabella raw, experiment ->', exp_name.upper())
    print('\\begin{footnotesize}')
    print('\\renewcommand{\\arraystretch}{1.25}')
    print(sae_analysis.style.to_latex(hrules = True, clines = 'skip-last;data')) 
    print('\\renewcommand{\\arraystretch}{1}')
    print('\\end{footnotesize}')
    


### Visualize accuracy and computational performance (figure generation)

In [ ]:
def visualize_comparison_compressed(act_name : str, act_param_id, figure_id : str = ''):

    # Generate figure and define colormap
    fig, axs = plt.subplots(2, 2, figsize = (14,9))
    axs = axs.reshape((-1,))
    cmap = plt.get_cmap('tab10')

    for idx_axs, exp_name in enumerate(('gaussian', 'shield', 'rod')):

        # Define conditionals
        cond_exp = (analysis['experiment'] == exp_name)
        cond_act = (analysis['act_name'] == act_name)
        
        # Extraction
        curr_analysis = analysis[cond_exp * cond_act]
        act_param = np.unique(curr_analysis['act_parameter'])[act_param_id]
        cond_actparam = curr_analysis['act_parameter'] == act_param
        curr_analysis = curr_analysis[cond_actparam]
        
        # Extract monitored histories and model name 
        standard = list(curr_analysis[curr_analysis['initialization'] == 'standard']['monitors'])
        eys = list(curr_analysis[curr_analysis['initialization'] == 'eys']['monitors'])
        model_name = list(curr_analysis[curr_analysis['initialization'] == 'eys']['model_name'])

        # Plot Training dynamics
        for init_item in (standard, eys):
            for idx in range(len(init_item)):
                epochs_range = np.arange(1, len(init_item[idx]['loss_val']) + 1)
                loss_val = np.minimum.accumulate(init_item[idx]['loss_val'])
                axs[idx_axs].loglog(
                    epochs_range,
                    loss_val,
                    linestyle = '--' if init_item == standard else '-', 
                    color = cmap(idx), 
                    linewidth = 2,
                    label = None if init_item == standard else model_name[idx].upper()
                )
        axs[idx_axs].grid(
            color = 'gray', which = 'both', linestyle = ':', linewidth = 0.3
        )
        axs[idx_axs].tick_params(
            axis = 'both', which = 'both', colors = 'gray', labelsize = 14
        )
        axs[idx_axs].set_xlabel('epoch', fontsize = 15)
        axs[idx_axs].set_ylabel(
            '$\min_{k \leq epoch} MSE_{val,normalized}(k)$', fontsize = 15
        )
        def get_curr_title(exp_name):
            if exp_name == 'gaussian':
                return 'PGA'
            elif exp_name == 'rod':
                return 'ROD'
            elif exp_name == 'shield':
                return 'ELS'
            else:
                raise ValueError
        axs[idx_axs].set_title(
            get_curr_title(exp_name), fontsize = 16, fontweight='bold', pad = 5
        )

    # Extract info on computational efficiency of training
    efficiency_hist = dict()
    experiment_names = np.unique(np.array(analysis['experiment']))
    model_names = np.unique(np.array(curr_analysis['model_name']))
    for model_name in model_names:
        efficiency_hist[model_name] = list()
    for exp_name in experiment_names:
        cond_exp = (analysis['experiment'] == exp_name)
        cond_act = (analysis['act_name'] == act_name)
        curr_analysis = analysis[cond_exp * cond_act]
        act_param_unique = np.unique(curr_analysis['act_parameter'])
        cond_act_param = (curr_analysis['act_parameter'] == \
                          act_param_unique[act_param_id])
        curr_analysis = curr_analysis[cond_act_param]
        for model_name in model_names:
            cond_stat = (curr_analysis['model_name'] == model_name)
            curr_eff = np.mean(curr_analysis[cond_stat]['training_efficiency'])
            efficiency_hist[model_name].append(curr_eff)

    # Setup grid
    axs[-1].grid(
        color = 'gray', 
        which = 'both', 
        axis = 'y', 
        linestyle = ':', 
        linewidth = 0.3
    )

    width = 0.15
    exp_range = np.arange(len(experiment_names))
    offsets = width * np.arange(len(model_names))

    for bar_idx, key in enumerate(efficiency_hist.keys()):
        axs[-1].bar(
            exp_range + offsets[bar_idx],
            efficiency_hist[key],
            width,
            label = key.upper()
        )

    # Set up legend, ticks, labels
    axs[-1].legend(ncols = 4, fontsize = 13, columnspacing = 1.3)
    axs[-1].set_ylabel('[s/epoch]', fontsize = 14)
    axs[-1].set_xticks(
        exp_range + 1.5 * width,
        [get_curr_title(exp_name) for exp_name in experiment_names]
    )
    axs[-1].tick_params(axis = 'both', labelsize = 13)
    plt.yscale(value = 'log')
    axs[-1].set_ylim([4e-2, 1.9e1])
    axs[-1].set_title('Training cost', fontsize = 16, pad = 5)
    axs[-1].tick_params(axis = 'y', which = 'both', colors = 'grey')

    # Adjust subplots
    plt.subplots_adjust(wspace = 0.4, hspace = 0.55)

    # Save figures
    savepath = os.path.join('..', 'results', 'figures')
    if not os.path.exists(savepath):
        os.makedirs(savepath)
    plt.savefig(
        os.path.join(savepath, 'comparison' + figure_id + '.png'), 
        bbox_inches='tight'
    )

In [ ]:
# Call function for visualization
visualize_comparison_compressed(act_name = 'leaky', act_param_id = 0)
visualize_comparison_compressed(
    act_name = 'leaky', act_param_id = 1, figure_id = '_suppl1'
)
visualize_comparison_compressed(
    act_name = 'hypact', act_param_id = 0, figure_id = '_suppl2'
)
visualize_comparison_compressed(
    act_name = 'hypact', act_param_id = 1, figure_id = '_suppl3'
)